In [31]:
#Zespoół smerfy 
#Cezary Pendras 121674
#Dominika Niedźwiecka 121542


# F1 2026 MIAMI GRAND PRIX – TOP-5 RACE PREDICTION
# Method : Weighted Multi-Criteria Decision Analysis (MCDA)
import numpy as np
import pandas as pd

# Normalisation

def minmax(series: pd.Series, higher_is_better: bool = True) -> pd.Series:
    """Min-max normalise a Series to [0, 1], direction-aware."""
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series([0.5] * len(series), index=series.index)
    norm = (series - lo) / (hi - lo)
    return norm if higher_is_better else (1.0 - norm)

#  DATA BLOCK 1 – 2026 CHAMPIONSHIP POINTS ENTERING MIAMI

championship_points_2026 = {
    # Driver                  Points
    "Kimi Antonelli":          72,   # 2× race wins (China, Japan) + sprint pts
    "George Russell":          63,   # 1× race win (Australia) + sprint pts
    "Charles Leclerc":         49,   # 3× podiums (P3, P4, P3)
    "Lewis Hamilton":          41,   # 3× top-5 (P4, P3, P6)
    "Lando Norris":            22,   # P5 Australia, DNS China, P5 Japan
    "Oscar Piastri":           18,   # DNS Australia, DNS China, P2 Japan
    "Oliver Bearman":          14,   # Haas – P5 China + other pts
    "Pierre Gasly":            10,   # Alpine – consistent midfield
    "Esteban Ocon":             6,
    "Liam Lawson":              5,
    "Arvid Lindblad":           4,
    "Max Verstappen":           8,   # P6 Australia, DNF China, P8 Japan
    "Isack Hadjar":             4,
    "Gabriel Bortoleto":        2,
    "Franco Colapinto":         1,
    "Carlos Sainz":             0,
    "Alexander Albon":          0,
    "Nico Hulkenberg":          0,
    "Fernando Alonso":          0,
    "Lance Stroll":             0,
    "Sergio Perez":             0,
    "Valtteri Bottas":          0,
}

In [32]:
#  DATA BLOCK 2 – 2026 SEASON AVERAGE FINISHING POSITION
#  Australia R1, China R2, Japan R3.
#  DNS / DNF / unclassified = coded as P20 (worst-case penalty)/# (20 = DNF / DNS / not classified)

results_2026 = {
    "Kimi Antonelli":  [2,  1,  1],
    "George Russell":  [1,  2,  4],
    "Charles Leclerc": [3,  4,  3],
    "Lewis Hamilton":  [4,  3,  6],
    "Lando Norris":    [5, 20,  5],
    "Oscar Piastri":   [20, 20, 2],
    "Oliver Bearman":  [7,  5, 20],   # crashed out in Japan
    "Pierre Gasly":    [10, 6,  7],
    "Esteban Ocon":    [11, 14, 10],
    "Liam Lawson":     [20, 7,  9],
    "Arvid Lindblad":  [8, 12, 14],
    "Max Verstappen":  [6, 20,  8],
    "Isack Hadjar":    [20, 8, 12],
    "Gabriel Bortoleto":[9, 20, 13],
    "Franco Colapinto":[20, 10, 20],
    "Carlos Sainz":    [20, 9, 20],
    "Alexander Albon": [20, 11, 20],
    "Nico Hulkenberg": [20, 11, 20],
    "Fernando Alonso": [20, 20, 20],
    "Lance Stroll":    [20, 20, 20],
    "Sergio Perez":    [20, 15, 20],
    "Valtteri Bottas": [20, 13, 20],
}

avg_finish_2026 = {
    driver: np.mean(finishes)
    for driver, finishes in results_2026.items()
}

In [33]:
#  DATA BLOCK 3 – HISTORICAL MIAMI GP AVERAGE FINISHING POSITION (2022–2025)
#  2022: 1-Verstappen, 2-Leclerc, 3-Carlos Sainz, 4-Valtteri Bottas, 5-Norris
#  2023: 1-Verstappen, 2-Alonso, 3-Leclerc, 4-Hamilton, 5-Bearman (not in F1)
#  2024: 1-Norris, 2-Verstappen, 3-Russell, 4-Antonelli (not in F1), 5-Hamilton
#  2025: 1-Piastri, 2-Norris, 3-Russell, 4-Verstappen, 5-Albon
#  Drivers who did not race that year are coded as P15.

miami_history = {
    # Driver             [2022, 2023, 2024, 2025]
    "Max Verstappen":    [1,    1,    2,    4],
    "Charles Leclerc":   [2,    3,    6,    7],
    "Carlos Sainz":      [3,    5,    7,    9],
    "Lando Norris":      [5,    8,    1,    2],
    "Oscar Piastri":     [15,   15,   8,    1],
    "Lewis Hamilton":    [15,   4,    5,    8],
    "George Russell":    [15,   15,   3,    3],
    "Kimi Antonelli":    [15,   15,   15,   6], 
    "Oliver Bearman":    [15,   15,   15,   15],
    "Pierre Gasly":      [18,   15,   15,   15],
    "Esteban Ocon":      [15,   15,   15,   15],
    "Liam Lawson":       [15,   15,   15,   15],
    "Arvid Lindblad":    [15,   15,   15,   15],
    "Max Verstappen":    [1,    1,    2,    4],
    "Isack Hadjar":      [15,   15,   15,   15],
    "Gabriel Bortoleto": [15,   15,   15,   18],
    "Franco Colapinto":  [15,   15,   15,   15],
    "Alexander Albon":   [15,   15,   18,   5],
    "Nico Hulkenberg":   [15,   15,   15,   15],
    "Fernando Alonso":   [15,   2,    15,   15],
    "Lance Stroll":      [15,   15,   15,   15],
    "Sergio Perez":      [15,   15,   15,   15],
    "Valtteri Bottas":   [4,    15,   15,   15],
}

avg_miami_hist = {
    driver: np.mean(results)
    for driver, results in miami_history.items()
}

In [34]:
#  DATA BLOCK 4 – TEAM CAR / UPGRADE PERFORMANCE SCORE (0 – 10)
team_upgrade_score = {
    # Driver                 Team upgrade score (0–10)
    "Kimi Antonelli":        8.5,   # Mercedes
    "George Russell":        8.5,   # Mercedes
    "Charles Leclerc":       8.0,   # Ferrari
    "Lewis Hamilton":        8.0,   # Ferrari
    "Lando Norris":          9.5,   # McLaren – "completely new car"
    "Oscar Piastri":         9.5,   # McLaren – "completely new car"
    "Oliver Bearman":        6.0,   # Haas (Ferrari PU)
    "Pierre Gasly":          5.5,   # Alpine (Mercedes PU)
    "Esteban Ocon":          5.5,   # Haas (Ferrari PU)
    "Liam Lawson":           5.0,   # Racing Bulls (Red Bull-Ford)
    "Arvid Lindblad":        5.0,   # Racing Bulls (Red Bull-Ford)
    "Max Verstappen":        8.5,   # Red Bull – significant rear wing test
    "Isack Hadjar":          8.5,   # Red Bull
    "Gabriel Bortoleto":     4.0,   # Audi
    "Franco Colapinto":      5.5,   # Alpine (Mercedes PU)
    "Carlos Sainz":          5.5,   # Williams (Mercedes PU)
    "Alexander Albon":       5.5,   # Williams (Mercedes PU)
    "Nico Hulkenberg":       4.0,   # Audi
    "Fernando Alonso":       4.5,   # Aston Martin (Honda)
    "Lance Stroll":          4.5,   # Aston Martin (Honda)
    "Sergio Perez":          4.5,   # Cadillac (Ferrari PU)
    "Valtteri Bottas":       4.5,   # Cadillac (Ferrari PU)
}

In [35]:
#  DATA BLOCK 5 – MIAMI 2026 QUALIFYING POSITION (official post-penalty grid)
miami_2026_quali = {
    "Kimi Antonelli":    1,
    "Max Verstappen":    2,
    "Charles Leclerc":   3,
    "Lando Norris":      4,
    "George Russell":    5,
    "Lewis Hamilton":    6,
    "Oscar Piastri":     7,
    "Franco Colapinto":  8,
    "Pierre Gasly":      9,
    "Nico Hulkenberg":   10,
    "Liam Lawson":       11,
    "Oliver Bearman":    12,
    "Carlos Sainz":      13,
    "Esteban Ocon":      14,
    "Alexander Albon":   15,
    "Arvid Lindblad":    16,
    "Fernando Alonso":   17,
    "Lance Stroll":      18,
    "Valtteri Bottas":   19,
    "Sergio Perez":      20,
    "Gabriel Bortoleto": 21,
    "Isack Hadjar":      22,   # starts from pit lane (DSQ)
}

In [36]:
#  MASTER DATAFRAME

all_drivers = list(championship_points_2026.keys())

df = pd.DataFrame(index=all_drivers)

df["champ_pts"]       = pd.Series(championship_points_2026)
df["avg_finish_2026"] = pd.Series(avg_finish_2026)
df["avg_miami_hist"]  = pd.Series(avg_miami_hist)
df["team_upgrade"]    = pd.Series(team_upgrade_score)
df["quali_pos"]       = pd.Series(miami_2026_quali)

# Fill missing values
df["champ_pts"]       = df["champ_pts"].fillna(0)
df["avg_finish_2026"] = df["avg_finish_2026"].fillna(15)
df["avg_miami_hist"]  = df["avg_miami_hist"].fillna(15)
df["team_upgrade"]    = df["team_upgrade"].fillna(4.5)
df["quali_pos"]       = df["quali_pos"].fillna(22)   # pit-lane / no time = last

#  NORMALISE ALL FEATURES to [0, 1]
#  Note: finishing/qualifying positions → lower = better → higher_is_better=False

df["n_champ_pts"]       = minmax(df["champ_pts"],       higher_is_better=True)
df["n_avg_finish_2026"] = minmax(df["avg_finish_2026"], higher_is_better=False)
df["n_avg_miami_hist"]  = minmax(df["avg_miami_hist"],  higher_is_better=False)
df["n_team_upgrade"]    = minmax(df["team_upgrade"],    higher_is_better=True)
df["n_quali_pos"]       = minmax(df["quali_pos"],       higher_is_better=False)


#  COMPUTE WEIGHTED COMPOSITE SCORE
#  S = 0.18·F̂1 + 0.14·F̂2 + 0.14·F̂3 + 0.14·F̂4 + 0.40·F̂5
#
#  Weights derived by: removing original F5 (15%), adding qualifying (40%),
#  then scaling remaining F1–F4 proportionally so total = 100 %.
#  Original F1–F4 sum = 85 % → scaled to 60 %: 25/85×60=18 %, 20/85×60≈14 % each.

WEIGHTS = {
    "n_champ_pts":       0.15,   # Championship points (2026)
    "n_avg_miami_hist":  0.15,   # Historical Miami avg finish
    "n_team_upgrade":    0.15,   # Team upgrade/car score
    "n_quali_pos":       0.55,   # Miami 2026 qualifying position  ← NEW
}

df["composite_score"] = sum(
    df[feat] * weight for feat, weight in WEIGHTS.items()
)

# Sort by composite score descending
df_sorted = df.sort_values("composite_score", ascending=False)
top5 = df_sorted.head(5).index.tolist()

#  RESULTS

def team_of(driver: str) -> str:
    teams = {
        "Kimi Antonelli":   "Mercedes",
        "George Russell":   "Mercedes",
        "Charles Leclerc":  "Ferrari",
        "Lewis Hamilton":   "Ferrari",
        "Lando Norris":     "McLaren-Mercedes",
        "Oscar Piastri":    "McLaren-Mercedes",
        "Oliver Bearman":   "Haas-Ferrari",
        "Pierre Gasly":     "Alpine-Mercedes",
        "Esteban Ocon":     "Haas-Ferrari",
        "Liam Lawson":      "Racing Bulls-Red Bull Ford",
        "Arvid Lindblad":   "Racing Bulls-Red Bull Ford",
        "Max Verstappen":   "Red Bull-Red Bull Ford",
        "Isack Hadjar":     "Red Bull-Red Bull Ford",
        "Gabriel Bortoleto":"Audi",
        "Franco Colapinto": "Alpine-Mercedes",
        "Carlos Sainz":     "Williams-Mercedes",
        "Alexander Albon":  "Williams-Mercedes",
        "Nico Hulkenberg":  "Audi",
        "Fernando Alonso":  "Aston Martin-Honda",
        "Lance Stroll":     "Aston Martin-Honda",
        "Sergio Perez":     "Cadillac-Ferrari",
        "Valtteri Bottas":  "Cadillac-Ferrari",
    }
    return teams.get(driver, "Unknown")

print("  " + "─" * 71)
print(f"  │ {'#':<2} │ {'Driver':<23} │ {'Team':<24} │ {'Score':>6} │  {'Grid':>4} │")
print("  " + "─" * 71)
for pos, (driver, row) in enumerate(df_sorted.head(10).iterrows(), start=1):
    marker = "◄── TOP 5" if pos <= 5 else ""
    grid_pos = int(miami_2026_quali.get(driver, 22))
    print(f"  │ {pos:<2} │ {driver:<23} │ {team_of(driver):<24} │ {row['composite_score']:.4f} │  P{grid_pos:<2} │  {marker}")
print("  " + "─" * 71)

#  Feature contribution breakdown for the top 5
print("\n  FEATURE BREAKDOWN FOR TOP-5 DRIVERS")
print("  (normalised feature scores, higher = better in all columns)\n")
cols = {
    "n_champ_pts":       "ChampPts",
    "n_avg_finish_2026": "AvgFin26",
    "n_avg_miami_hist":  "MiamiHist",
    "n_team_upgrade":    "Upgrade",
    "n_quali_pos":       "Quali26",
    "composite_score":   "TOTAL",
}
header = f"  {'Driver':<24}" + "".join(f"  {v:>9}" for v in cols.values())
print(header)
print("  " + "─" * (len(header) - 2))
for driver in top5:
    row = df_sorted.loc[driver]
    vals = "".join(f"  {row[k]:>9.4f}" for k in cols.keys())
    print(f"  {driver:<24}{vals}")


  ───────────────────────────────────────────────────────────────────────
  │ #  │ Driver                  │ Team                     │  Score │  Grid │
  ───────────────────────────────────────────────────────────────────────
  │ 1  │ Kimi Antonelli          │ Mercedes                 │ 0.8555 │  P1  │  ◄── TOP 5
  │ 2  │ Charles Leclerc         │ Ferrari                  │ 0.8315 │  P3  │  ◄── TOP 5
  │ 3  │ Max Verstappen          │ Red Bull-Red Bull Ford   │ 0.8132 │  P2  │  ◄── TOP 5
  │ 4  │ Lando Norris            │ McLaren-Mercedes         │ 0.7954 │  P4  │  ◄── TOP 5
  │ 5  │ George Russell          │ Mercedes                 │ 0.7729 │  P5  │  ◄── TOP 5
  │ 6  │ Lewis Hamilton          │ Ferrari                  │ 0.6981 │  P6  │  
  │ 7  │ Oscar Piastri           │ McLaren-Mercedes         │ 0.6458 │  P7  │  
  │ 8  │ Franco Colapinto        │ Alpine-Mercedes          │ 0.4178 │  P8  │  
  │ 9  │ Pierre Gasly            │ Alpine-Mercedes          │ 0.4022 │  P9  │  
  │ 10 │